# Land, river, and sea-ice initial conditions for RTENSO2026: year selection

No model restarts exist for 2026, so the ELM (land), MOSART (river), and
MPAS-Seaice initial conditions come from one year of the BruteForce archive,
used unchanged (averaging restarts would break the consistency of snow layers,
ice categories, and phenology). Two branches choose the year:

| Branch | Year selected | Target |
| --- | --- | --- |
| `clim` | closest to the climatology | candidate-year mean |
| `obs` | closest to the observed state | ERA5 anomaly on the target date |

The 2015 dates are done the same way, with 2015 excluded from the candidates,
to test hindcast sensitivity against the actual 2015 restarts, and to check
how close each branch's choice is to the actual 2015 state.

## Distance metric

For key field $f$ (section 1), candidate-year values are converted to
standardized anomalies $z_{y,i}=(x_{y,i}-\bar x_i)/S$, where
$S^2=\sum_i w_i s_i^2/\sum_i w_i$ is the area-weighted mean interannual
variance over the candidate years. The distance of year $y$ from a target
$t$ is

$$d_f(y)=\sqrt{\frac{\sum_i w_i\,(z_{y,i}-t_i)^2}{\sum_i w_i}}$$

- `clim`: $t=0$. $d_f$ is the RMS anomaly relative to the RMS spread; its mean
  square over the candidate years is 1, so $d_f<1$ is closer than a typical year.
- `obs`: $t$ is the ERA5 standardized anomaly on the target date,
  $(o_T-\bar o)/S_\mathrm{obs}$, with the ERA5 mean and spread over the same
  candidate years and date. Comparing anomalies cancels model-ERA5 biases and
  unit or layer-depth differences. Only fields with an ERA5 counterpart are
  used.

The river (MOSART) is not in the distance (its storage is driven by land
runoff); its restart always comes from the land year. The two soil-water
fields cover disjoint layers (1-3 and 4+), so no layer is counted twice.

Component distance $D_c=\sqrt{\operatorname{mean}_{f\in c} d_f^2}$; joint
distance $D=\sqrt{\sum_c a_c D_c^2/\sum_c a_c}$ with component weights $a_c$
(default 1). `MODE = "joint"` takes the year with the smallest $D$ for all
components (one coupled state); `"per_component"` takes each component's
smallest $D_c$.

The BruteForce archive is read-only here. Code: `util/clim_restart.py`.

In [ ]:
from pathlib import Path
import importlib
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import netCDF4 as nc

SEARCH_ROOTS = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
HELPER_RELATIVE_PATHS = (
    Path("util"),
    Path("rtenso2026/util"),
    Path("ensemble_init/rtenso2026/util"),
    Path("e3sm_s2d_proc/ensemble_init/rtenso2026/util"),
)
HELPER_DIR = next(
    (
        root / relative_path
        for root in SEARCH_ROOTS
        for relative_path in HELPER_RELATIVE_PATHS
        if (root / relative_path / "clim_restart.py").is_file()
    ),
    None,
)
if HELPER_DIR is None:
    raise FileNotFoundError("Could not find util/clim_restart.py from the notebook directory.")
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

import clim_restart
clim_restart = importlib.reload(clim_restart)
from clim_restart import (
    compute_distances,
    distance_to_model_year,
    era5_available,
    make_plan,
    metric_table,
    select_years,
    source_file,
    summarize_restart,
    write_selected_restarts,
)

warnings.filterwarnings("ignore", category=RuntimeWarning)

## Configuration

In [ ]:
# Initial-condition archive (opened read-only).
SRC_ROOT = Path("/global/cfs/cdirs/e3sm/zhan391/v3.LR.S2D.ENSINT/BruteForce")
SRC_PREFIX = "20251024_s2d_spinup"

# Fixed RTENSO2026 output location: OUT_ROOT/YYYYMMDD/<branch>/<prefix>.<stream>.<date>.nc
OUT_ROOT = Path("/global/cfs/cdirs/e3sm/S2S2D/RTENSO2026/initial_conditions")
OUT_PREFIX = "20251024_s2d_spinup"   # matches my_refcase2 in the hindcast driver

BRANCHES = ["clim", "obs"]
COMPONENTS = ["elm", "mosart", "mpassi"]
TEST_DATES = ["2015-02-01", "2015-05-01", "2015-08-01", "2015-11-01"]
HINDCAST_DATES = ["2026-02-01", "2026-05-01", "2026-08-01", "2026-11-01"]
TARGET_DATES = TEST_DATES + HINDCAST_DATES

# Candidate years. None uses every available year (1980-2018).
FIRST_YEAR = None
LAST_YEAR = None
EXCLUDE_YEARS = []
EXCLUDE_TARGET_YEAR = True          # 2015 cannot select itself

# Compare area-weighted means over (dlat, dlon)-degree boxes; None = model points.
# 30 x 60 matched the actual 2015 state best (README, "Choice of box size").
BOX = (30, 60)
MODE = "joint"                      # or "per_component"
COMPONENT_WEIGHTS = {}              # e.g. {"mpassi": 2.0}; default 1 each

# Model and ERA5 key fields are cached under the output root.
CACHE_DIR = OUT_ROOT / "cache"

WRITE_OUTPUT = False                # True copies the selected restarts to OUT_ROOT
OVERWRITE = False

## 1. Key fields and their ERA5 counterparts

In [ ]:
display(metric_table())

## 2. Distances and selection

The first run for a calendar month reads the model key fields (about 3.5
minutes) and, for `obs`, the ERA5 fields (one 00Z slice per variable and year);
both are cached. `obs` is skipped for dates ERA5 does not cover yet.

In [ ]:
plans, distances, selections = {}, {}, {}
for branch in BRANCHES:
    for date in TARGET_DATES:
        if branch == "obs" and not era5_available(int(date[:4]), date[5:], COMPONENTS):
            print(f"skip obs {date}: ERA5 not available yet")
            continue
        plan = make_plan(
            date, SRC_ROOT, OUT_ROOT, branch=branch, components=COMPONENTS,
            src_prefix=SRC_PREFIX, out_prefix=OUT_PREFIX,
            first_year=FIRST_YEAR, last_year=LAST_YEAR,
            exclude_years=EXCLUDE_YEARS, exclude_target_year=EXCLUDE_TARGET_YEAR,
        )
        table = compute_distances(plan, box=BOX, component_weights=COMPONENT_WEIGHTS, cache_dir=CACHE_DIR)
        key = (branch, date)
        plans[key], distances[key] = plan, table
        selections[key] = select_years(table, COMPONENTS, MODE)

summary = []
for (branch, date), table in distances.items():
    ranked = table["joint"].sort_values()
    row = {"branch": branch, "target_date": date, "n_candidates": len(table)}
    row.update({f"{c} year": y for c, y in selections[(branch, date)].items()})
    row.update({"best D": ranked.iloc[0], "2nd year": ranked.index[1], "2nd D": ranked.iloc[1],
                "3rd year": ranked.index[2], "3rd D": ranked.iloc[2]})
    summary.append(row)
summary = pd.DataFrame(summary)
display(summary.round(3))

In [ ]:
# Ten closest years for each branch and date (field, component, and joint distances).
for (branch, date), table in distances.items():
    print(f"{branch} {date}")
    display(table.sort_values("joint").head(10).round(3))

In [ ]:
# Joint distance by year for each branch; the selected year is red.
fig, axes = plt.subplots(len(BRANCHES), len(TARGET_DATES), figsize=(4 * len(TARGET_DATES), 3.2 * len(BRANCHES)),
                         sharey=True, squeeze=False)
for r, branch in enumerate(BRANCHES):
    for c, date in enumerate(TARGET_DATES):
        ax = axes[r, c]
        if (branch, date) not in distances:
            ax.set_title(f"{branch} {date}: no ERA5")
            ax.axis("off")
            continue
        table = distances[(branch, date)]
        best = int(table["joint"].idxmin())
        years = table.index.to_numpy()
        ax.bar(years, table["joint"], color=["tab:red" if y == best else "0.6" for y in years])
        for comp, style in zip([c for c in COMPONENTS if c in table], ("-", "--", ":")):
            ax.plot(years, table[comp], style, lw=1, label=comp)
        ax.axhline(1, color="k", lw=0.5)
        ax.set_title(f"{branch} {date}: {best}")
    axes[r, 0].set_ylabel(f"{branch} distance")
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
# Field distances of the ten closest years for one branch and date.
KEY = ("obs", TARGET_DATES[0])
table = distances[KEY].sort_values("joint").head(10)
fields = [c for c in table.columns if ": " in c]
fig, ax = plt.subplots(figsize=(10, 0.45 * len(fields) + 1.5))
im = ax.imshow(table[fields].T, aspect="auto", cmap="viridis_r", vmin=0, vmax=2)
ax.set_xticks(range(len(table)), table.index)
ax.set_yticks(range(len(fields)), fields)
ax.set_title(f"Field distances d_f, ten closest years, {KEY[0]} {KEY[1]}")
fig.colorbar(im, ax=ax, label="d_f")
fig.tight_layout()
plt.show()

## 3. Write the selected restarts

Copies each selected restart (unchanged) to
`OUT_ROOT/YYYYMMDD/<branch>/<prefix>.<stream>.<target date>.nc`, writes
`rpointer.lnd`, `rpointer.rof`, and `rpointer.ice`, and records the selection
and full distance table in `selection.json`. Set `WRITE_OUTPUT = True` to run.

In [ ]:
if WRITE_OUTPUT:
    for key, plan in plans.items():
        write_selected_restarts(plan, selections[key], distances[key], mode=MODE, box=BOX,
                                component_weights=COMPONENT_WEIGHTS, overwrite=OVERWRITE)

rows = []
for (branch, date), plan in plans.items():
    for comp in COMPONENTS:
        year = selections[(branch, date)][comp]
        out, src = plan.output(comp), plan.source(comp, year)
        rows.append({"branch": branch, "target_date": date, "component": comp, "year": year,
                     "written": out.is_file(),
                     "same size as source": out.is_file() and out.stat().st_size == src.stat().st_size})
display(pd.DataFrame(rows))

## 4. 2015 check: how close is each branch's year to the actual 2015 state?

For the 2015 dates the actual model restarts exist, so every candidate year can
be scored against them with the same metric (target = the actual 2015
standardized anomaly, on the fields both branches can see: the `obs` fields,
at the same `BOX` scale). A good `obs` selection should rank near the top; the
rank correlation shows how well each branch's distance tracks the true
closeness to 2015.

In [ ]:
truth = {}
check = []
for date in [d for d in TARGET_DATES if d.startswith("2015")]:
    ref_plan = plans.get(("clim", date)) or next(p for (b, d), p in plans.items() if d == date)
    truth[date] = distance_to_model_year(ref_plan, 2015, box=BOX, observable_only=True,
                                         component_weights=COMPONENT_WEIGHTS, cache_dir=CACHE_DIR)
    true_rank = truth[date]["joint"].rank()
    row = {"target_date": date, "closest to actual 2015": int(truth[date]["joint"].idxmin()),
           "its distance": truth[date]["joint"].min()}
    for branch in BRANCHES:
        if (branch, date) not in distances:
            continue
        year = int(distances[(branch, date)]["joint"].idxmin())
        row[f"{branch} year"] = year
        row[f"{branch}: distance to 2015"] = truth[date].loc[year, "joint"]
        row[f"{branch}: rank"] = int(true_rank[year])
        row[f"{branch}: rank corr"] = distances[(branch, date)]["joint"].rank().corr(true_rank)
    check.append(row)
display(pd.DataFrame(check).round(3))

In [ ]:
# Global summaries: each branch's selected year against the actual 2015 restarts.
compare_rows = []
for date in [d for d in TARGET_DATES if d.startswith("2015")]:
    for comp in COMPONENTS:
        actual = summarize_restart(source_file(SRC_ROOT, SRC_PREFIX, comp, 2015, date[5:]), comp)
        chosen = {}
        for branch in BRANCHES:
            if (branch, date) in selections:
                year = selections[(branch, date)][comp]
                chosen[branch] = (year, summarize_restart(source_file(SRC_ROOT, SRC_PREFIX, comp, year, date[5:]), comp))
        for name in actual:
            row = {"target_date": date, "component": comp, "quantity": name, "actual_2015": actual[name]}
            for branch, (year, values) in chosen.items():
                row[f"{branch} year"] = year
                row[branch] = values[name]
            compare_rows.append(row)
display(pd.DataFrame(compare_rows).round(3))

In [ ]:
# Sea-ice concentration: actual 2015 and each branch's year minus 2015.
DATE = "2015-02-01"

def ice_conc(year):
    with nc.Dataset(source_file(SRC_ROOT, SRC_PREFIX, "mpassi", year, DATE[5:])) as ds:
        return (np.degrees(ds["latCell"][:]), np.degrees(ds["lonCell"][:]),
                ds["iceAreaCategory"][0, :, :, 0].sum(axis=1))

lat, lon, actual = ice_conc(2015)
panels = [("actual 2015", actual, "Blues", 0, 1)]
for branch in BRANCHES:
    if (branch, DATE) in selections:
        year = selections[(branch, DATE)]["mpassi"]
        panels.append((f"{branch} {year} - 2015", ice_conc(year)[2] - actual, "RdBu_r", -0.5, 0.5))

fig, axes = plt.subplots(2, len(panels), figsize=(5 * len(panels), 9),
                         subplot_kw={"projection": "polar"}, squeeze=False)
for row, (hemi, sel, sign) in enumerate((("NH", lat > 45, 1), ("SH", lat < -45, -1))):
    theta, r = np.radians(lon[sel]), 90 - sign * lat[sel]
    for col, (title, field, cmap, vmin, vmax) in enumerate(panels):
        ax = axes[row, col]
        ax.set_facecolor("0.8")  # land (no ocean cells)
        sc = ax.scatter(theta, r, c=field[sel], s=0.2, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_ylim(0, 45)
        ax.set_yticklabels([])
        ax.set_title(f"{hemi} {title}")
        fig.colorbar(sc, ax=ax, shrink=0.7)
fig.suptitle(f"Sea-ice concentration, {DATE}")
fig.tight_layout()
plt.show()

In [ ]:
# ELM snow water equivalent (H2OSNO) on grid cells: actual 2015 and each branch's year minus 2015.
DATE = "2015-02-01"

def gridcell_swe(year):
    with nc.Dataset(source_file(SRC_ROOT, SRC_PREFIX, "elm", year, DATE[5:])) as ds:
        ds.set_auto_mask(False)
        swe = ds["H2OSNO"][:]
        wt = ds["cols1d_wtxy"][:] * (ds["cols1d_active"][:] > 0)
        idx = ds["cols1d_gridcell_index"][:] - 1
        glon, glat = ds["grid1d_lon"][:], ds["grid1d_lat"][:]
    num = np.bincount(idx, weights=np.where(np.abs(swe) < 1e35, swe, 0) * wt, minlength=glon.size)
    den = np.bincount(idx, weights=wt, minlength=glon.size)
    return glon, glat, np.where(den > 0, num / np.where(den > 0, den, 1), np.nan)

glon, glat, actual = gridcell_swe(2015)
panels = [("actual 2015", actual, "viridis", 0, 300)]
for branch in BRANCHES:
    if (branch, DATE) in selections:
        year = selections[(branch, DATE)]["elm"]
        panels.append((f"{branch} {year} - 2015", gridcell_swe(year)[2] - actual, "RdBu_r", -100, 100))

fig, axes = plt.subplots(1, len(panels), figsize=(6 * len(panels), 4.5), squeeze=False)
for ax, (title, field, cmap, vmin, vmax) in zip(axes[0], panels):
    sc = ax.scatter(glon, glat, c=field, s=1, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(f"H2OSNO (mm) {title}")
    ax.set_ylim(-60, 90)
    fig.colorbar(sc, ax=ax, shrink=0.8)
fig.suptitle(f"ELM snow water equivalent, {DATE}")
fig.tight_layout()
plt.show()